# 📊 Sales Explorer Notebook

Query the PostgreSQL `demo` database and visualize sales data with matplotlib.

**Prerequisites:** The Sales Explorer DB must be seeded (happens automatically on container startup).

In [ ]:
import psycopg2
import matplotlib
import matplotlib.pyplot as plt

# Use a dark style for charts
plt.style.use('dark_background')
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['font.size'] = 11

# Connect to PostgreSQL (trust auth, no password needed)
conn = psycopg2.connect(dbname='demo', user='coder', host='localhost')
cur = conn.cursor()
print('✅ Connected to demo database')

## Revenue by Category

Bar chart showing total revenue per product category.

In [ ]:
cur.execute("""
    SELECT category, SUM(total_amount) AS revenue
    FROM sales
    GROUP BY category
    ORDER BY revenue DESC
""")
rows = cur.fetchall()
categories = [r[0] for r in rows]
revenues   = [float(r[1]) for r in rows]

colors = ['#38bdf8', '#f472b6', '#34d399', '#fb923c', '#818cf8']
fig, ax = plt.subplots()
bars = ax.bar(categories, revenues, color=colors, edgecolor='none', width=0.6)
ax.set_title('Total Revenue by Category', fontweight='bold', pad=15)
ax.set_ylabel('Revenue ($)')
ax.bar_label(bars, fmt='$%.0f', padding=5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Monthly Sales Trend

Line chart showing revenue over time, grouped by month.

In [ ]:
cur.execute("""
    SELECT TO_CHAR(sale_date, 'YYYY-MM') AS month,
           SUM(total_amount)             AS revenue,
           SUM(quantity)                 AS qty
    FROM sales
    GROUP BY month
    ORDER BY month
""")
rows = cur.fetchall()
months  = [r[0] for r in rows]
rev     = [float(r[1]) for r in rows]
qty     = [int(r[2]) for r in rows]

fig, ax1 = plt.subplots()
ax1.fill_between(months, rev, alpha=0.3, color='#38bdf8')
ax1.plot(months, rev, color='#38bdf8', linewidth=2, marker='o', markersize=5, label='Revenue')
ax1.set_ylabel('Revenue ($)', color='#38bdf8')
ax1.tick_params(axis='y', labelcolor='#38bdf8')

ax2 = ax1.twinx()
ax2.plot(months, qty, color='#f472b6', linewidth=2, marker='s', markersize=5, linestyle='--', label='Quantity')
ax2.set_ylabel('Quantity', color='#f472b6')
ax2.tick_params(axis='y', labelcolor='#f472b6')

ax1.set_title('Monthly Sales Trend', fontweight='bold', pad=15)
plt.xticks(rotation=45)
fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.95))
ax1.spines['top'].set_visible(False)
plt.tight_layout()
plt.show()

## Revenue by Region (Pie Chart)

In [ ]:
cur.execute("""
    SELECT region, SUM(total_amount) AS revenue
    FROM sales
    GROUP BY region
    ORDER BY revenue DESC
""")
rows = cur.fetchall()
regions  = [r[0] for r in rows]
revenues = [float(r[1]) for r in rows]

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    revenues, labels=regions, autopct='%1.1f%%',
    colors=['#38bdf8', '#34d399', '#fb923c', '#818cf8'],
    startangle=140, pctdistance=0.8,
    wedgeprops=dict(width=0.45, edgecolor='#0f172a', linewidth=2)
)
for t in autotexts:
    t.set_fontsize(10)
ax.set_title('Revenue by Region', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## Custom Query

Try your own WHERE clause! Edit the `where_clause` variable below.

In [ ]:
# ✏️ Edit this WHERE clause to filter data
where_clause = "category = 'Electronics'"

query = f"""
    SELECT product, SUM(total_amount) AS revenue, SUM(quantity) AS qty
    FROM sales
    WHERE {where_clause}
    GROUP BY product
    ORDER BY revenue DESC
"""
print(f'Running: SELECT ... FROM sales WHERE {where_clause} ...')
cur.execute(query)
rows = cur.fetchall()

products = [r[0] for r in rows]
revenues = [float(r[1]) for r in rows]
qtys     = [int(r[2]) for r in rows]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#38bdf8', '#f472b6', '#34d399', '#fb923c', '#818cf8', '#facc15']
ax1.barh(products, revenues, color=colors[:len(products)], edgecolor='none', height=0.5)
ax1.set_xlabel('Revenue ($)')
ax1.set_title(f'Revenue — {where_clause}', fontweight='bold')
ax1.invert_yaxis()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.barh(products, qtys, color=colors[:len(products)], edgecolor='none', height=0.5)
ax2.set_xlabel('Quantity Sold')
ax2.set_title(f'Quantity — {where_clause}', fontweight='bold')
ax2.invert_yaxis()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Clean up
cur.close()
conn.close()
print('Connection closed.')